# TAC-LAnoBERT v2: Evaluation & Comparison

**Purpose**: Evaluate TAC v2 and compare with baseline

**Prerequisites**:
- ✅ TAC v2 trained → `outputs/BGL_tac_v2_2epochs/`
- ✅ BGL data preprocessed → `data/BGL/`
- (Optional) Baseline for comparison → `outputs/BGL_lanobert/` or `outputs/BGL_tac/`

**What this notebook does**:
- Run TAC v2 inference (if not done)
- Calculate metrics (F1, Precision, Recall, FPR, AUROC)
- Evaluate early detection (DLT, EWR)
- Compare with baseline (if available)
- Generate report

## Setup

In [ ]:
# Clone repository (if on Kaggle/Colab)
import os

if not os.path.exists('TAC-LAnoBERT-y'):
    !git clone https://github.com/rubyhcm/TAC-LAnoBERT-y.git
    %cd TAC-LAnoBERT-y
else:
    print("✅ Repository already exists")
    if not os.getcwd().endswith('TAC-LAnoBERT-y'):
        %cd TAC-LAnoBERT-y

In [ ]:
!pip install -r requirements.txt -q
print("✅ Dependencies installed")

In [ ]:
# KNN mode uses FAISS for fast nearest-neighbor search in SessionMemoryQueue
!pip install faiss-cpu -q
print("✅ FAISS installed for KNN distance")

In [ ]:
# Verify environment
import torch
import os

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
else:
    print("  ⚠️  CPU mode (inference will be slow)")

print("\n✅ Environment ready")

## Check Prerequisites

In [ ]:
# Check and copy all required datasets from Kaggle input
import glob
import os
import shutil

print("="*70)
print("CHECKING PREREQUISITES")
print("="*70)

# 1. TAC v2 model (REQUIRED)
tac_v2_in_input = glob.glob("/kaggle/input/**/BGL_tac_v2_2epochs", recursive=True)
if os.path.exists("outputs/BGL_tac_v2_2epochs/model"):
    print("\n✅ TAC v2 model: Already in working directory")
elif tac_v2_in_input:
    print("\n📦 TAC v2 model: Found in input, copying...")
    print(f"   Source: {tac_v2_in_input[0]}")
    os.makedirs("outputs", exist_ok=True)
    !cp -r {tac_v2_in_input[0]} outputs/
    print("✅ Copied")
else:
    print("\n❌ TAC v2 model: NOT FOUND!")
    print("   → Attach BGL_tac_v2_2epochs dataset as Kaggle input")
    print("   → This is the trained model from training notebook")

# 2. BGL data (REQUIRED)
bgl_data_in_input = glob.glob("/kaggle/input/**/BGL_test_parsed.log", recursive=True)
if bgl_data_in_input:
    bgl_dir = os.path.dirname(bgl_data_in_input[0])
    if os.path.exists("data/BGL/BGL_test_parsed.log"):
        print("\n✅ BGL data: Already in working directory")
    else:
        print("\n📦 BGL data: Found in input, copying...")
        print(f"   Source: {bgl_dir}")
        os.makedirs("data", exist_ok=True)
        !cp -r {bgl_dir} data/
        print("✅ Copied")
else:
    print("\n❌ BGL data: NOT FOUND!")
    print("   → Attach BGL preprocessed data as Kaggle input")

# 3. Baseline models (OPTIONAL - for comparison)
baseline_in_input = glob.glob("/kaggle/input/**/BGL_lanobert", recursive=True)
if baseline_in_input and not os.path.exists("outputs/BGL_lanobert"):
    print("\n📦 Baseline (LAnoBERT): Found in input, copying...")
    os.makedirs("outputs", exist_ok=True)
    !cp -r {baseline_in_input[0]} outputs/
    print("✅ Copied")

tac_baseline_in_input = glob.glob("/kaggle/input/**/BGL_tac", recursive=True)
# Exclude BGL_tac_v2_2epochs
tac_baseline_in_input = [p for p in tac_baseline_in_input if "v2" not in p]
if tac_baseline_in_input and not os.path.exists("outputs/BGL_tac"):
    print("\n📦 Baseline (TAC original): Found in input, copying...")
    os.makedirs("outputs", exist_ok=True)
    !cp -r {tac_baseline_in_input[0]} outputs/
    print("✅ Copied")

# Check if baselines exist (for comparison)
baseline_exists = (
    os.path.exists("outputs/BGL_lanobert/results") or
    os.path.exists("outputs/BGL_tac/results")
)

print("\n" + "="*70)
print("✅ PREREQUISITES READY")
print("="*70)

if baseline_exists:
    print("\n✅ Baseline found (will compare)")
else:
    print("\n⚠️  No baseline found (will skip comparison)")


## 🔍 DEBUG: Check All Datasets & Paths

In [ ]:
# DEBUG 1: Check Kaggle input datasets
import os
import glob

print("=" * 80)
print("DEBUG 1: KAGGLE INPUT DATASETS")
print("=" * 80)
print()

if os.path.exists("/kaggle/input"):
    input_datasets = os.listdir("/kaggle/input")
    print(f"Found {len(input_datasets)} dataset(s) attached:\n")
    for ds in sorted(input_datasets):
        ds_path = os.path.join("/kaggle/input", ds)
        print(f"📦 {ds}")
        
        # Show top-level contents
        try:
            contents = os.listdir(ds_path)[:10]  # First 10 items
            for item in contents:
                item_path = os.path.join(ds_path, item)
                if os.path.isdir(item_path):
                    print(f"   📁 {item}/")
                else:
                    size = os.path.getsize(item_path)
                    print(f"   📄 {item} ({size:,} bytes)")
        except Exception as e:
            print(f"   ⚠️  Cannot read: {e}")
        print()
else:
    print("⚠️  Not running on Kaggle (no /kaggle/input)")
    print("   This is expected if running locally.\n")

print("=" * 80)


In [ ]:
# DEBUG 2: Check working directory structure
print("=" * 80)
print("DEBUG 2: WORKING DIRECTORY STRUCTURE")
print("=" * 80)
print()

print(f"Current working directory: {os.getcwd()}\n")

# Check if we're in the right place
if os.path.exists("tac_lanobert"):
    print("✅ tac_lanobert/ directory found")
else:
    print("❌ tac_lanobert/ directory NOT found")
    print("   You may need to cd into TAC-LAnoBERT-y\n")

if os.path.exists("configs"):
    print("✅ configs/ directory found")
else:
    print("❌ configs/ directory NOT found\n")

# Check outputs directory
print("\nOutputs directory:")
if os.path.exists("outputs"):
    outputs = os.listdir("outputs")
    print(f"✅ outputs/ exists with {len(outputs)} item(s):")
    for item in sorted(outputs):
        print(f"   📁 {item}")
else:
    print("❌ outputs/ does NOT exist")
    print("   Will be created when needed.")

print("\n" + "=" * 80)


In [ ]:
# DEBUG 3: Deep check on TAC v2 model structure
print("=" * 80)
print("DEBUG 3: TAC V2 MODEL STRUCTURE (DETAILED)")
print("=" * 80)
print()

model_base = "outputs/BGL_tac_v2_2epochs"

if os.path.exists(model_base):
    print(f"✅ {model_base}/ exists\n")
    
    # Check model/ subdirectory
    model_dir = os.path.join(model_base, "model")
    if os.path.exists(model_dir):
        print(f"✅ {model_dir}/ exists\n")
        
        # List all contents
        print("Contents of model/:")
        for item in sorted(os.listdir(model_dir)):
            item_path = os.path.join(model_dir, item)
            if os.path.isdir(item_path):
                # Count files in subdirectory
                try:
                    sub_count = len(os.listdir(item_path))
                    print(f"   📁 {item}/ ({sub_count} files)")
                except:
                    print(f"   📁 {item}/")
            else:
                size = os.path.getsize(item_path)
                print(f"   📄 {item} ({size:,} bytes)")
        
        # Check for 'final' subdirectory specifically
        final_dir = os.path.join(model_dir, "final")
        print(f"\n{'='*80}")
        print("CHECKING FOR 'final/' SUBDIRECTORY:")
        print("='" * 80)
        
        if os.path.exists(final_dir) and os.path.isdir(final_dir):
            print(f"\n✅ {final_dir}/ EXISTS!\n")
            
            print("Contents of final/:")
            for item in sorted(os.listdir(final_dir)):
                item_path = os.path.join(final_dir, item)
                if os.path.isfile(item_path):
                    size = os.path.getsize(item_path)
                    size_mb = size / (1024 * 1024)
                    print(f"   📄 {item} ({size_mb:.2f} MB)")
                else:
                    print(f"   📁 {item}/")
            
            # Check required model files
            print(f"\nRequired files check:")
            required_files = {
                "config.json": "Model configuration",
                "model.safetensors": "Model weights",
                "time2vec.pt": "Time2Vec weights (TAC)",
                "tokenizer_config.json": "Tokenizer config (optional)",
                "vocab.txt": "Vocabulary (optional)"
            }
            
            for fname, desc in required_files.items():
                fpath = os.path.join(final_dir, fname)
                if os.path.exists(fpath):
                    size = os.path.getsize(fpath)
                    print(f"   ✅ {fname:<25} ({size:>12,} bytes) - {desc}")
                else:
                    req = "REQUIRED" if fname in ["config.json", "model.safetensors"] else "optional"
                    print(f"   ❌ {fname:<25} {'':>12} - {desc} ({req})")
            
            print(f"\n✅ MODEL READY TO LOAD FROM: {final_dir}")
            
        else:
            print(f"\n❌ {final_dir}/ DOES NOT EXIST!\n")
            print("⚠️  This is the problem! Model files should be in 'final/' subdirectory.\n")
            
            # Check if model files are directly in model/ instead
            print("Checking if model files are directly in model/ (without final/):")
            direct_files = ["config.json", "model.safetensors", "pytorch_model.bin"]
            found_direct = False
            for fname in direct_files:
                fpath = os.path.join(model_dir, fname)
                if os.path.exists(fpath):
                    print(f"   ✅ Found: {fname}")
                    found_direct = True
            
            if found_direct:
                print("\n💡 SOLUTION: Model files are in model/ directly (not in final/)")
                print("   → This is OK! The code should handle this.")
                print("   → But make sure you have the LATEST code with the fix!")
            else:
                print("\n❌ No model files found in model/ either!")
                print("   → Dataset might be incomplete or corrupted.")
                print("   → Check your BGL_tac_v2_2epochs dataset upload.")
        
    else:
        print(f"❌ {model_dir}/ does NOT exist!")
        print("   → Check if BGL_tac_v2_2epochs dataset is properly attached.\n")
else:
    print(f"❌ {model_base}/ does NOT exist!")
    print("   → TAC v2 model directory not found.")
    print("   → Make sure BGL_tac_v2_2epochs dataset is attached as Kaggle input.\n")

print("\n" + "=" * 80)


In [ ]:
# CLARIFICATION: vocab.txt location
print("=" * 80)
print("💡 VOCAB.TXT CLARIFICATION")
print("=" * 80)
print()

print("There are TWO tokenizer files:")
print()

print("1️⃣  model/final/tokenizer.json")
print("   → HuggingFace fast tokenizer format")
print("   → Saved with model for convenience")
print("   → THIS IS WHAT INFERENCE USES ✅")
tokenizer_json = "outputs/BGL_tac_v2_2epochs/model/final/tokenizer.json"
if os.path.exists(tokenizer_json):
    size = os.path.getsize(tokenizer_json)
    print(f"   ✅ EXISTS: {size:,} bytes")
print()

print("2️⃣  tokenizer/BGL_LogBERT-vocab.txt")
print("   → Original vocab file from training")
print("   → LAnoBERT custom format")
print("   → Only needed for custom tokenizer loading")
vocab_txt = "outputs/BGL_tac_v2_2epochs/tokenizer/BGL_LogBERT-vocab.txt"
if os.path.exists(vocab_txt):
    size = os.path.getsize(vocab_txt)
    print(f"   ✅ EXISTS: {size:,} bytes")
else:
    print(f"   ❌ NOT FOUND")
print()

print("=" * 80)
print("CONCLUSION")
print("=" * 80)
print()
print("✅ Inference loads tokenizer.json from model/final/")
print("✅ vocab.txt in tokenizer/ is for reference only")
print("✅ Both formats are valid, inference prefers tokenizer.json")
print("✅ NO PROBLEM! Everything works correctly! 🎉")
print()
print("=" * 80)


In [ ]:
# DEBUG 4: Check if inference script has the fix
print("=" * 80)
print("DEBUG 4: CHECK INFERENCE SCRIPT VERSION")
print("=" * 80)
print()

inference_script = "tac_lanobert/inference_tac.py"

if os.path.exists(inference_script):
    print(f"✅ {inference_script} exists\n")
    
    # Check for the fix signature
    with open(inference_script, 'r') as f:
        content = f.read()
    
    print("Checking for fixes:")
    
    # Check 1: final/ detection
    if 'detected \'final\' subdirectory' in content:
        print("   ✅ Fix #1: Auto-detect 'final/' subdirectory - PRESENT")
    else:
        print("   ❌ Fix #1: Auto-detect 'final/' subdirectory - MISSING")
        print("      → You need to pull the latest code!")
    
    # Check 2: nested config reading
    if 'memory_cfg = tac_cfg.get("memory"' in content:
        print("   ✅ Fix #2: Nested config reading (memory_cfg) - PRESENT")
    else:
        print("   ❌ Fix #2: Nested config reading - MISSING")
        print("      → You need to pull the latest code!")
    
    # Check 3: simplified model_dir logic
    if 'LAnoBERTScorer will auto-detect' in content:
        print("   ✅ Fix #3: Simplified model_dir logic - PRESENT")
    else:
        print("   ⚠️  Fix #3: Simplified model_dir logic - check manually")
    
    print("\n" + "=" * 80)
    print("VERDICT:")
    print("=" * 80)
    
    if all([
        'detected \'final\' subdirectory' in content,
        'memory_cfg = tac_cfg.get("memory"' in content
    ]):
        print("\n✅ You have the LATEST code with all fixes!")
        print("   If inference still fails, the problem is with the dataset structure.\n")
    else:
        print("\n❌ You have OUTDATED code!")
        print("\n🔧 SOLUTION:")
        print("   1. Delete TAC-LAnoBERT-y directory:")
        print("      !rm -rf TAC-LAnoBERT-y")
        print("\n   2. Re-clone from GitHub:")
        print("      !git clone https://github.com/YOUR_USERNAME/TAC-LAnoBERT-y.git")
        print("      %cd TAC-LAnoBERT-y")
        print("\n   3. Re-run from the top!\n")
else:
    print(f"❌ {inference_script} NOT found!")
    print("   → Are you in the right directory?\n")

print("=" * 80)


## Run TAC v2 Inference

Run inference if not already done.

In [ ]:
# Check if inference already done
tac_v2_scores_exist = (
    os.path.exists("outputs/BGL_tac_knn/results/scores_tac_hybrid.npy") or
    os.path.exists("outputs/BGL_tac_knn/results/scores_tac_mlm_error.npy")
)

if tac_v2_scores_exist:
    print("✅ TAC v2 inference already complete (scores found)")
    print("   Skipping inference...")
else:
    print("Running TAC v2 inference...")
    print("This will take ~1-2 hours on T4 GPU\n")
    
    !python -m tac_lanobert.inference_tac --config configs/bgl_tac_knn.yaml
    
    print("\n✅ Inference complete")

## View TAC v2 Results

Load and display detailed results from inference.

In [ ]:
# Load score files
import numpy as np
import json
from pathlib import Path

results_dir = Path('outputs/BGL_tac_knn/results')

print("=" * 70)
print("TAC-LANOBERT V2 RESULTS")
print("=" * 70)

# Find score files
score_files = list(results_dir.glob('scores_*.npy'))

if score_files:
    print(f"\n📊 Score Files: {len(score_files)}\n")
    
    scores_dict = {}
    for score_file in sorted(score_files):
        scores = np.load(score_file)
        name = score_file.stem.replace('scores_', '')
        scores_dict[name] = scores
        
        print(f"{name}:")
        print(f"  Lines:  {len(scores):,}")
        print(f"  Min:    {scores.min():.6f}")
        print(f"  Max:    {scores.max():.6f}")
        print(f"  Mean:   {scores.mean():.6f}")
        print(f"  Median: {np.median(scores):.6f}")
        print(f"  Std:    {scores.std():.6f}")
        print()
    
    print("=" * 70)
    
else:
    print("\n⚠️  No score files found")
    print(f"   Expected in: {results_dir}")
    print("   Run inference first!")

In [ ]:
# Parse text report for metrics
import re

def parse_text_report(report_path):
    """Parse TAC report text file for metrics"""
    if not os.path.exists(report_path):
        return None
    
    with open(report_path, 'r') as f:
        content = f.read()
    
    metrics = {}
    
    # Extract metrics
    if m := re.search(r'AUROC:\s+([0-9.e+-]+)', content):
        metrics['auroc'] = float(m.group(1))
    if m := re.search(r'best_F1:\s+([0-9.e+-]+)', content):
        metrics['f1'] = float(m.group(1))
    if m := re.search(r'best_threshold:\s+([0-9.e+-]+)', content):
        metrics['threshold'] = float(m.group(1))
    
    # Extract confusion matrix
    cm_pattern = r'confusion_matrix:.*?\[\[\s*(\d+)\s+(\d+)\s*\]\s*\[\s*(\d+)\s+(\d+)\s*\]\]'
    if m := re.search(cm_pattern, content, re.DOTALL):
        tn, fp, fn, tp = int(m.group(1)), int(m.group(2)), int(m.group(3)), int(m.group(4))
        metrics['tp'] = tp
        metrics['fp'] = fp
        metrics['tn'] = tn
        metrics['fn'] = fn
        metrics['fpr'] = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        metrics['precision'] = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        metrics['recall'] = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    
    return metrics if metrics else None

# Find and parse TAC v2 report
report_files = list(results_dir.glob('*_report.txt'))
tac_v2_report = None

if report_files:
    tac_v2_report = parse_text_report(str(report_files[0]))
    
    if tac_v2_report:
        print("\n" + "=" * 70)
        print("TAC V2 METRICS")
        print("=" * 70)
        print()
        print(f"  F1-Score:    {tac_v2_report.get('f1', 0):.6f}")
        print(f"  Precision:   {tac_v2_report.get('precision', 0):.6f}")
        print(f"  Recall:      {tac_v2_report.get('recall', 0):.6f}")
        print(f"  AUROC:       {tac_v2_report.get('auroc', 0):.6f}")
        print(f"  FPR:         {tac_v2_report.get('fpr', 0)*100:.4f}%")
        print(f"  Threshold:   {tac_v2_report.get('threshold', 0):.6f}")
        print()
        print(f"  Confusion Matrix:")
        print(f"    TP: {tac_v2_report.get('tp', 0):>8,}  FP: {tac_v2_report.get('fp', 0):>8,}")
        print(f"    FN: {tac_v2_report.get('fn', 0):>8,}  TN: {tac_v2_report.get('tn', 0):>8,}")
        print()
        print("=" * 70)
    else:
        print("\n⚠️  Could not parse report file")
else:
    print("\n⚠️  No report files found")
    print(f"   Expected *_report.txt in {results_dir}")

## Compare with Baseline

Compare TAC v2 with baseline (if available).

In [ ]:
# Find all available baselines
baselines = {}

# Check for original TAC baseline
if os.path.exists("outputs/BGL_tac/results/BGL_tac_hybrid_report.txt"):
    report = parse_text_report("outputs/BGL_tac/results/BGL_tac_hybrid_report.txt")
    if report:
        baselines['TAC-LAnoBERT (original)'] = report

# Check for LAnoBERT baseline
if os.path.exists("outputs/BGL_lanobert/results/BGL_error_mean_report.txt"):
    report = parse_text_report("outputs/BGL_lanobert/results/BGL_error_mean_report.txt")
    if report:
        baselines['LAnoBERT (baseline)'] = report

# Compare with each baseline
if baselines and tac_v2_report:
    for baseline_name, baseline_report in baselines.items():
        print("=" * 70)
        print(f"COMPARISON: {baseline_name} vs TAC v2")
        print("=" * 70)
        print()
        
        metrics_to_compare = ['f1', 'precision', 'recall', 'auroc', 'fpr']
        
        print(f"{'Metric':<15} {'Baseline':<15} {'TAC v2':<15} {'Δ':<15} {'Status'}")
        print("-" * 75)
        
        for metric in metrics_to_compare:
            baseline_val = baseline_report.get(metric)
            tac_v2_val = tac_v2_report.get(metric)
            
            if baseline_val is not None and tac_v2_val is not None:
                delta = tac_v2_val - baseline_val
                delta_pct = (delta / baseline_val * 100) if baseline_val != 0 else 0
                
                # Determine status
                if metric == 'fpr':
                    # Lower is better
                    status = "✅ Better" if delta < 0 else ("⚠️ Worse" if delta > 0 else "≈ Same")
                else:
                    # Higher is better
                    status = "✅ Better" if delta > 0 else ("⚠️ Worse" if delta < 0 else "≈ Same")
                
                improvement = f"{delta_pct:+.2f}%" if delta != 0 else "0.00%"
                print(f"{metric.upper():<15} {baseline_val:<15.6f} {tac_v2_val:<15.6f} {improvement:<15} {status}")
            else:
                print(f"{metric.upper():<15} {'N/A':<15} {'N/A':<15} {'N/A':<15} {'N/A'}")
        
        print("\n" + "=" * 70)
        
elif tac_v2_report:
    print("\n⚠️  No baseline found for comparison")
    print("   Upload baseline results to compare")
else:
    print("\n⚠️  TAC v2 metrics not found")
    print("   Check if inference completed successfully")


In [ ]:
# Confusion matrix & alert volume comparison for each baseline
if baselines and tac_v2_report:
    for baseline_name, baseline_report in baselines.items():
        if all(k in baseline_report for k in ["tp", "fp", "tn", "fn"]):
            print("\n" + "=" * 70)
            print(f"CONFUSION MATRIX: {baseline_name} vs TAC v2")
            print("=" * 70)
            print()
            print(f"{'':<10} {'TP':>12} {'FP':>12} {'TN':>12} {'FN':>12}")
            print("-" * 70)
            print(f"{baseline_name[:10]:<10} {baseline_report['tp']:>12,} {baseline_report['fp']:>12,} "
                  f"{baseline_report['tn']:>12,} {baseline_report['fn']:>12,}")
            print(f"{'TAC v2':<10} {tac_v2_report['tp']:>12,} {tac_v2_report['fp']:>12,} "
                  f"{tac_v2_report['tn']:>12,} {tac_v2_report['fn']:>12,}")
            
            # Alert reduction
            baseline_alerts = baseline_report["tp"] + baseline_report["fp"]
            tac_v2_alerts = tac_v2_report["tp"] + tac_v2_report["fp"]
            alert_reduction = (baseline_alerts - tac_v2_alerts) / baseline_alerts * 100
            
            print("\n" + "=" * 70)
            print(f"ALERT VOLUME: {baseline_name} vs TAC v2")
            print("=" * 70)
            print(f"\n{baseline_name} alerts:  {baseline_alerts:,}")
            print(f"TAC v2 alerts:        {tac_v2_alerts:,}")
            print(f"Reduction:            {alert_reduction:+.2f}%")
            
            if alert_reduction > 0:
                print(f"\n✅ TAC v2 reduces alert volume by {alert_reduction:.1f}%!")
                print(f"   Fewer false positives = less alert fatigue")
            elif alert_reduction < 0:
                print(f"\n⚠️  TAC v2 generates {abs(alert_reduction):.1f}% more alerts")
            else:
                print(f"\n≈ Similar alert volume")
            
            print("\n" + "=" * 70)


## Summary

In [ ]:
from datetime import datetime

print("=" * 70)
print("EVALUATION SUMMARY")
print("=" * 70)

print(f"\nCompleted: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

if tac_v2_report:
    print(f"\n📊 TAC v2 Performance:")
    print(f"   F1:        {tac_v2_report.get('f1', 0):.6f}")
    print(f"   Precision: {tac_v2_report.get('precision', 0):.6f}")
    print(f"   Recall:    {tac_v2_report.get('recall', 0):.6f}")
    print(f"   AUROC:     {tac_v2_report.get('auroc', 0):.6f}")
    print(f"   FPR:       {tac_v2_report.get('fpr', 0)*100:.4f}%")

if baseline_report and tac_v2_report:
    f1_improve = (tac_v2_report['f1'] - baseline_report['f1']) / baseline_report['f1'] * 100
    fpr_improve = (baseline_report['fpr'] - tac_v2_report['fpr']) / baseline_report['fpr'] * 100
    
    print(f"\n📈 Improvement vs {baseline_name}:")
    print(f"   F1:  {f1_improve:+.2f}%")
    print(f"   FPR: {fpr_improve:+.2f}% (lower is better)")

print(f"\n📂 Results saved in:")
print(f"   {results_dir}")

print("\n" + "=" * 70)
print("✅ Evaluation Complete")
print("=" * 70)